In [1]:
# Make the notebook behave as if it were running from the repo root.
# evaluation.py uses a relative data path, so this matters.
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loading import feature_groups
from evaluation import SEED, gini_normalized, load_train

train = load_train()
categorical_cols, quantity_cols = feature_groups(train.columns)

# SAME column order as src/train_baseline_models.py. Their preprocessor selects
# columns by POSITION, not name, so this ordering is load-bearing, not cosmetic.
X = train[categorical_cols + quantity_cols]
y = train["target"]

print(f"rows {len(train):,} | categorical {len(categorical_cols)} | quantity {len(quantity_cols)}")
print(f"claim rate {y.mean()*100:.4f}%")

rows 475,967 | categorical 31 | quantity 26
claim rate 3.6685%


In [2]:
# Fit the same forest as the baseline, ONCE, on the whole training split.
# The baseline script fits it five times (once per fold). For importances one
# fit is enough — and keeping it in memory means we can examine it repeatedly
# without refitting.
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline

from train_baseline_models import _build_preprocessor

t0 = time.perf_counter()
forest = make_pipeline(
    _build_preprocessor(len(categorical_cols), len(quantity_cols), scale=False),
    RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                           n_jobs=-1, random_state=SEED),
)
forest.fit(X, y)
print(f"fitted in {time.perf_counter() - t0:.1f}s")

pre = forest.named_steps["columntransformer"]
rf = forest.named_steps["randomforestclassifier"]
print(f"encoded columns: {len(rf.feature_importances_)}  (from {X.shape[1]} real columns)")

fitted in 91.9s
encoded columns: 226  (from 57 real columns)


In [3]:
def gain_importance_by_column(preprocessor, model, feature_cols):
    """
    Map one-hot dummy importances back to their parent column and sum them.

    Needed because the forest sees 226 columns, not 57: each categorical column
    was expanded into one column per value, so its importance arrives split
    into that many pieces.
    """
    encoded = preprocessor.get_feature_names_out()
    importances = model.feature_importances_
    assert len(encoded) == len(importances), "name/importance length mismatch"

    parents = []
    for name in encoded:
        body = name.split("__", 1)[1]        # strip the 'cat__' / 'qty__' prefix
        if name.startswith("cat__"):
            body = body.rsplit("_", 1)[0]    # strip the trailing '_<value>'
        parents.append(body)

    out = (pd.DataFrame({"column": parents, "gain": importances})
             .groupby("column", as_index=False)["gain"].sum())
    out["dummies"] = pd.Series(parents).value_counts().reindex(out["column"]).values
    out["group"] = out["column"].str.extract(r"ps_(ind|reg|car|calc)_")[0]
    out = out.sort_values("gain", ascending=False).reset_index(drop=True)
    out.index += 1                            # so the index reads as a rank

    missing = set(feature_cols) - set(out["column"])
    assert not missing, f"columns lost in mapping: {sorted(missing)}"
    assert np.isclose(out["gain"].sum(), 1.0), f"gains sum to {out['gain'].sum():.6f}"
    return out


imp = gain_importance_by_column(pre, rf, categorical_cols + quantity_cols)
print(f"mapped {len(rf.feature_importances_)} encoded columns -> {len(imp)} real columns\n")
imp.head(15)

mapped 226 encoded columns -> 57 real columns



,column,gain,dummies,group
1,ps_car_13,0.069996,1,car
2,ps_reg_03,0.057903,1,reg
3,ps_car_14,0.043648,1,car
4,ps_ind_03,0.034611,1,ind
5,ps_reg_02,0.033609,1,reg
6,ps_ind_15,0.032718,1,ind
7,ps_ind_05_cat,0.030306,8,ind
8,ps_calc_02,0.028134,1,calc
9,ps_calc_01,0.027739,1,calc
10,ps_calc_03,0.027650,1,calc


In [4]:
# AC-3 first pass: which prefix groups carry the gain?
by_group = (imp.groupby("group")
              .agg(columns=("column", "size"),
                   total_gain=("gain", "sum"))
              .sort_values("total_gain", ascending=False))
by_group["gain_per_column"] = (by_group["total_gain"] / by_group["columns"]).round(5)
print(by_group.to_string())

print("\n--- how flat is the ranking? real signal spreads, filler clusters ---")
print(f"drop across ranks 1-7  : {imp.loc[1,'gain'] - imp.loc[7,'gain']:.5f}")
print(f"drop across ranks 8-15 : {imp.loc[8,'gain'] - imp.loc[15,'gain']:.5f}")

print("\n--- where the calc columns sit ---")
calc = imp[imp.group == "calc"]
print(f"{len(calc)} calc columns | ranks {calc.index.min()}-{calc.index.max()} "
      f"| total gain {calc.gain.sum():.4f}")
print(f"of the top 20, {(imp.head(20).group == 'calc').sum()} are calc")

print("\n--- the high-cardinality columns, after summing their dummies back ---")
print(imp[imp.dummies > 2].to_string())

print("\n--- the bottom 10 ---")
print(imp.tail(10).to_string())

       columns  total_gain  gain_per_column
group                                      
calc        20    0.340307          0.01702
car         16    0.336779          0.02105
ind         18    0.205626          0.01142
reg          3    0.117288          0.03910

--- how flat is the ranking? real signal spreads, filler clusters ---
drop across ranks 1-7  : 0.03969
drop across ranks 8-15 : 0.00316

--- where the calc columns sit ---
20 calc columns | ranks 8-51 | total gain 0.3403
of the top 20, 6 are calc

--- the high-cardinality columns, after summing their dummies back ---
           column      gain  dummies group
7   ps_ind_05_cat  0.030306        8   ind
16  ps_car_11_cat  0.024356      104   car
17  ps_car_01_cat  0.023696       13   car
20  ps_car_06_cat  0.023028       18   car
26  ps_car_03_cat  0.016986        3   car
27  ps_car_09_cat  0.016434        6   car
28  ps_car_05_cat  0.016302        3   car
33  ps_car_07_cat  0.014178        3   car
34  ps_ind_04_cat  0.013181  

In [6]:
# Definitions only — the cheap half of cell 5, so a kernel restart doesn't cost
# ten minutes of cross-validation to get back to a usable state.
cat_nocalc = [c for c in categorical_cols if "_calc_" not in c]
qty_nocalc = [c for c in quantity_cols if "_calc_" not in c]


def forest_for(n_cat, n_qty):
    """Identical to the baseline forest — only the column count differs."""
    return make_pipeline(
        _build_preprocessor(n_cat, n_qty, scale=False),
        RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                               n_jobs=-2, random_state=SEED),
    )


print(f"{len(cat_nocalc) + len(qty_nocalc)} columns without calc")

37 columns without calc


In [5]:
# AC-3: does the calc group carry signal? Test by removing it and measuring,
# not by summing gains — gain is exactly the measure we don't trust here.
from evaluation import cross_validate_model

cat_nocalc = [c for c in categorical_cols if "_calc_" not in c]
qty_nocalc = [c for c in quantity_cols if "_calc_" not in c]

print(f"with calc   : {len(categorical_cols) + len(quantity_cols)} columns "
      f"(cat {len(categorical_cols)}, qty {len(quantity_cols)})")
print(f"without calc: {len(cat_nocalc) + len(qty_nocalc)} columns "
      f"(cat {len(cat_nocalc)}, qty {len(qty_nocalc)})")
print(f"dropped     : {len(categorical_cols) + len(quantity_cols) - len(cat_nocalc) - len(qty_nocalc)}\n")


def forest_for(n_cat, n_qty):
    """Identical to the baseline forest — only the column count differs."""
    return make_pipeline(
        _build_preprocessor(n_cat, n_qty, scale=False),
        RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                               n_jobs=-1, random_state=SEED),
    )


t0 = time.perf_counter()

print("WITH calc  (should reproduce the 0.2687 in results.md)")
with_mean, with_std, with_scores = cross_validate_model(
    forest_for(len(categorical_cols), len(quantity_cols)),
    train[categorical_cols + quantity_cols], y)

print("\nWITHOUT calc")
without_mean, without_std, without_scores = cross_validate_model(
    forest_for(len(cat_nocalc), len(qty_nocalc)),
    train[cat_nocalc + qty_nocalc], y)

print(f"\nran in {(time.perf_counter() - t0)/60:.1f} min")

with calc   : 57 columns (cat 31, qty 26)
without calc: 37 columns (cat 25, qty 12)
dropped     : 20

WITH calc  (should reproduce the 0.2687 in results.md)
      fold 1  gini +0.26617  (claims in fold: 3,492)
      fold 2  gini +0.27499  (claims in fold: 3,493)
      fold 3  gini +0.26710  (claims in fold: 3,492)
      fold 4  gini +0.26380  (claims in fold: 3,492)
      fold 5  gini +0.27123  (claims in fold: 3,492)
      mean +0.26866  std 0.00397

WITHOUT calc
      fold 1  gini +0.26806  (claims in fold: 3,492)
      fold 2  gini +0.27496  (claims in fold: 3,493)
      fold 3  gini +0.27233  (claims in fold: 3,492)
      fold 4  gini +0.27034  (claims in fold: 3,492)
      fold 5  gini +0.27653  (claims in fold: 3,492)
      mean +0.27244  std 0.00305

ran in 10.2 min


In [5]:
delta = without_mean - with_mean
folds_won = int((without_scores > with_scores).sum())

print(f"with calc    (57 cols): {with_mean:+.5f} +/- {with_std:.5f}")
print(f"without calc (37 cols): {without_mean:+.5f} +/- {without_std:.5f}")
print(f"delta                 : {delta:+.5f}")
print(f"per-fold deltas       : {np.round(without_scores - with_scores, 5).tolist()}")
print(f"folds improved        : {folds_won} of {len(without_scores)}")

verdict = ("improvement" if delta > without_std
           else "regression" if delta < -without_std
           else "inconclusive")
print(f"\nverdict vs fold spread: {verdict.upper()}")

NameError: name 'without_mean' is not defined

In [7]:
# AC-2 groundwork: permutation importance on the 37-column model.
#
# Why permute the ORIGINAL columns rather than the encoded ones: shuffling a
# one-hot dummy would produce impossible rows (two values true at once). By
# shuffling the raw column and letting the whole pipeline re-run, both the
# encoding fragmentation and the cardinality bias drop out.
#
# Why this fold: we rebuild the harness's own 5-fold split and use fold 5, so
# these numbers sit on the same footing as the cross-validation scores.
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold
from evaluation import N_FOLDS

X_nc = train[cat_nocalc + qty_nocalc]
Xa, ya = X_nc.values, y.values

splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
tr_idx, va_idx = list(splitter.split(Xa, ya))[-1]
print(f"fit on {len(tr_idx):,} rows, permute on {len(va_idx):,} rows")

perm_model = forest_for(len(cat_nocalc), len(qty_nocalc))
perm_model.fit(Xa[tr_idx], ya[tr_idx])

baseline_gini = gini_normalized(ya[va_idx], perm_model.predict_proba(Xa[va_idx])[:, 1])
print(f"baseline gini on that fold: {baseline_gini:.5f}\n")


def gini_scorer(estimator, X, y):
    """permutation_importance needs a scorer taking (estimator, X, y)."""
    return gini_normalized(y, estimator.predict_proba(X)[:, 1])


t0 = time.perf_counter()
result = permutation_importance(
    perm_model, Xa[va_idx], ya[va_idx],
    scoring=gini_scorer, n_repeats=3, random_state=SEED, n_jobs=-2,
)
print(f"took {(time.perf_counter() - t0)/60:.1f} min for {Xa.shape[1]} columns\n")

perm = (pd.DataFrame({"column": X_nc.columns,
                      "gini_drop": result.importances_mean,
                      "drop_std": result.importances_std})
          .sort_values("gini_drop", ascending=False)
          .reset_index(drop=True))
perm.index += 1
perm["group"] = perm["column"].str.extract(r"ps_(ind|reg|car|calc)_")[0]

print(f"columns whose shuffling did NOT hurt (drop <= 0): "
      f"{int((perm.gini_drop <= 0).sum())} of {len(perm)}\n")
perm.head(15)

fit on 380,774 rows, permute on 95,193 rows
baseline gini on that fold: 0.27653

took 1.5 min for 37 columns

columns whose shuffling did NOT hurt (drop <= 0): 12 of 37



,column,gini_drop,drop_std,group
1,ps_ind_05_cat,0.027686,0.001788,ind
2,ps_ind_17_bin,0.013813,0.000998,ind
3,ps_car_13,0.013250,0.002222,car
4,ps_ind_03,0.012719,0.001205,ind
5,ps_reg_01,0.008407,0.001138,reg
6,ps_reg_02,0.007798,0.001932,reg
7,ps_car_07_cat,0.007544,0.001311,car
8,ps_reg_03,0.006951,0.001354,reg
9,ps_ind_15,0.004387,0.001773,ind
10,ps_car_04_cat,0.003759,0.000804,car


In [9]:
# AC-3 completed: what does each remaining group contribute?
# Ablate one group at a time from the 37-column model, on the same folds.
# Reference is the 37-column model recorded in results.md.
from evaluation import cross_validate_model

REFERENCE_MEAN, REFERENCE_STD = 0.27244, 0.00305

rows = []
for grp in ["reg", "car", "ind"]:
    cat_keep = [c for c in cat_nocalc if f"_{grp}_" not in c]
    qty_keep = [c for c in qty_nocalc if f"_{grp}_" not in c]
    n_dropped = 37 - len(cat_keep) - len(qty_keep)

    print("=" * 60)
    print(f"minus {grp}: dropping {n_dropped} columns, "
          f"{len(cat_keep) + len(qty_keep)} remain")
    print("=" * 60)

    t0 = time.perf_counter()
    m, s, scores = cross_validate_model(
        forest_for(len(cat_keep), len(qty_keep)),
        train[cat_keep + qty_keep], y)

    delta = m - REFERENCE_MEAN
    print(f"  delta vs 37-col reference: {delta:+.5f}")
    print(f"  cost per dropped column  : {delta / n_dropped:+.6f}")
    print(f"  ran in {(time.perf_counter() - t0)/60:.1f} min\n")

    rows.append({"dropped_group": grp, "n_dropped": n_dropped,
                 "columns_left": len(cat_keep) + len(qty_keep),
                 "gini_mean": round(m, 5), "gini_std": round(s, 5),
                 "delta": round(delta, 5)})

print("=" * 60)
ablation = pd.DataFrame(rows).sort_values("delta")
print(f"reference: 37 columns, {REFERENCE_MEAN:.5f} +/- {REFERENCE_STD:.5f}\n")
print(ablation.to_string(index=False))

minus reg: dropping 3 columns, 34 remain
      fold 1  gini +0.25550  (claims in fold: 3,492)
      fold 2  gini +0.26727  (claims in fold: 3,493)
      fold 3  gini +0.25871  (claims in fold: 3,492)
      fold 4  gini +0.25175  (claims in fold: 3,492)
      fold 5  gini +0.26008  (claims in fold: 3,492)
      mean +0.25866  std 0.00517
  delta vs 37-col reference: -0.01378
  cost per dropped column  : -0.004594
  ran in 4.9 min

minus car: dropping 16 columns, 21 remain
      fold 1  gini +0.22602  (claims in fold: 3,492)
      fold 2  gini +0.24191  (claims in fold: 3,493)
      fold 3  gini +0.22914  (claims in fold: 3,492)
      fold 4  gini +0.23822  (claims in fold: 3,492)
      fold 5  gini +0.24355  (claims in fold: 3,492)
      mean +0.23577  std 0.00698
  delta vs 37-col reference: -0.03667
  cost per dropped column  : -0.002292
  ran in 2.0 min

minus ind: dropping 18 columns, 19 remain
      fold 1  gini +0.20782  (claims in fold: 3,492)
      fold 2  gini +0.22064  (claims

In [10]:
# AC-2: examine the top ten against the target.
# Top ten taken from PERMUTATION importance, not gain — gain is the measure
# we showed to be misleading.
TOP_TEN = ["ps_ind_05_cat", "ps_ind_17_bin", "ps_car_13", "ps_ind_03",
           "ps_reg_01", "ps_reg_02", "ps_car_07_cat", "ps_reg_03",
           "ps_ind_15", "ps_car_04_cat"]

BASE_RATE = y.mean()
print(f"base claim rate: {BASE_RATE*100:.4f}%\n")


def examine(col, df, n_bins=10, raw_value_limit=20):
    """
    How the claim rate moves across one column's values.

    Missing (-1) always gets its OWN row — that is the point of the exercise.
    Columns with few distinct values are grouped by raw value; genuinely
    continuous ones are cut into deciles, since thousands of one-row groups
    would tell you nothing.
    """
    s, target = df[col], df["target"]
    base = target.mean()
    is_cat = col.endswith(("_cat", "_bin"))
    missing = s == -1

    if is_cat or s[~missing].nunique() <= raw_value_limit:
        grp = s.astype(str).replace("-1", "missing (-1)")
        mode = "raw values"
    else:
        grp = pd.Series(index=s.index, dtype=object)
        grp[missing] = "missing (-1)"
        grp[~missing] = ("q" + (pd.qcut(s[~missing], n_bins, labels=False,
                                        duplicates="drop") + 1).astype(str))
        mode = f"{n_bins} deciles"

    out = (pd.DataFrame({"value": grp, "y": target})
             .groupby("value", as_index=False)
             .agg(n=("y", "size"), claims=("y", "sum"), claim_rate=("y", "mean")))
    out["pct_of_rows"] = (out.n / len(df) * 100).round(2)
    out["lift"] = (out.claim_rate / base).round(3)
    out["claim_rate"] = (out.claim_rate * 100).round(3)
    out.attrs["mode"] = mode
    return out.sort_values("claim_rate", ascending=False).reset_index(drop=True)


rows = []
for c in TOP_TEN:
    t = examine(c, train)
    rows.append({
        "column": c,
        "kind": "categorical" if c.endswith(("_cat", "_bin")) else "quantity",
        "n_distinct": int(train[c].nunique()),
        "missing_pct": round((train[c] == -1).mean() * 100, 3),
        "lift_min": t.lift.min(),
        "lift_max": t.lift.max(),
        "lift_spread": round(t.lift.max() - t.lift.min(), 3),
    })

summary = pd.DataFrame(rows).sort_values("lift_spread", ascending=False)
summary

base claim rate: 3.6685%



,column,kind,n_distinct,missing_pct,lift_min,lift_max,lift_spread
9,ps_car_04_cat,categorical,10,0.000,0.583,1.964,1.381
0,ps_ind_05_cat,categorical,8,0.981,0.930,2.229,1.299
6,ps_car_07_cat,categorical,3,1.923,0.954,2.121,1.167
5,ps_reg_02,quantity,19,0.000,0.648,1.708,1.060
2,ps_car_13,quantity,64194,0.000,0.655,1.605,0.950
3,ps_ind_03,quantity,12,0.000,0.819,1.741,0.922
1,ps_ind_17_bin,categorical,2,0.000,0.927,1.527,0.600
7,ps_reg_03,quantity,4987,18.096,0.780,1.366,0.586
4,ps_reg_01,quantity,10,0.000,0.661,1.120,0.459
8,ps_ind_15,quantity,14,0.000,0.832,1.278,0.446


In [11]:
for c in TOP_TEN:
    t = examine(c, train)
    miss = (train[c] == -1).mean() * 100
    print("=" * 70)
    print(f"{c}   ({t.attrs['mode']}, {len(t)} groups, "
          f"{train[c].nunique()} distinct values, {miss:.2f}% missing)")
    print("=" * 70)
    print(t.to_string(index=False))
    print()

ps_ind_05_cat   (raw values, 8 groups, 8 distinct values, 0.98% missing)
       value      n  claims  claim_rate  pct_of_rows  lift
missing (-1)   4671     382       8.178         0.98 2.229
           2   3322     241       7.255         0.70 1.978
           6  16479     977       5.929         3.46 1.616
           4  14642     768       5.245         3.08 1.430
           1   6706     328       4.891         1.41 1.333
           5   1323      64       4.837         0.28 1.319
           3   6631     293       4.419         1.39 1.204
           0 422193   14408       3.413        88.70 0.930

ps_ind_17_bin   (raw values, 2 groups, 2 distinct values, 0.00% missing)
value      n  claims  claim_rate  pct_of_rows  lift
    1  57847    3241       5.603        12.15 1.527
    0 418120   14220       3.401        87.85 0.927

ps_car_13   (10 deciles, 10 groups, 64194 distinct values, 0.00% missing)
value     n  claims  claim_rate  pct_of_rows  lift
  q10 47597    2803       5.889        1

In [12]:
print(f"base claim rate {BASE_RATE*100:.4f}%\n")
print(f"{'column':16} {'lift':>6} {'rows':>8} {'claim%':>8}  treatment")
print("-" * 78)
for c in TOP_TEN:
    if (train[c] == -1).any():
        t = examine(c, train)
        r = t[t.value == "missing (-1)"].iloc[0]
        kept = c.endswith(("_cat", "_bin"))
        treatment = ("categorical -> missingness KEPT as its own column"
                     if kept else
                     "quantity -> missingness DESTROYED by median impute")
        print(f"{c:16} {r.lift:>6.3f} {r.n:>8,} {r.claim_rate:>8.3f}  {treatment}")

base claim rate 3.6685%

column             lift     rows   claim%  treatment
------------------------------------------------------------------------------
ps_ind_05_cat     2.229    4,671    8.178  categorical -> missingness KEPT as its own column
ps_car_07_cat     2.121    9,151    7.781  categorical -> missingness KEPT as its own column
ps_reg_03         0.780   86,133    2.861  quantity -> missingness DESTROYED by median impute
